In [ ]:
%pip install fastparquet

In [ ]:
# tiny_tydi_bigram_acc_loadstyle.py
import math, unicodedata as ud
from collections import Counter
import pandas as pd

BOS, EOS, OOV = "<s>", "</s>", "[oov]"

# --- Tokenize: NFKC + lowercase, drop punctuation, split on whitespace ---
def tok(s):
    if not isinstance(s, str): return []
    s = ud.normalize("NFKC", s).lower()
    s = "".join(ch for ch in s if not ud.category(ch).startswith("P"))
    return s.split()

# --- Train: vocab (train-only) + bigram counts with BOS/EOS ---
def fit(texts, min_freq=1):
    cnt = Counter(); [cnt.update(tok(t)) for t in texts]
    kept = sorted(w for w,c in cnt.items() if c>=min_freq and w not in {BOS,EOS,OOV})
    stoi = {OOV:0, BOS:1, EOS:2, **{w:i+3 for i,w in enumerate(kept)}}
    counts, row, oov = {}, {}, stoi[OOV]
    for t in texts:
        w = tok(t)
        if not w: continue
        ids = [stoi.get(x,oov) for x in [BOS, *w, EOS]]
        for A,B in zip(ids[:-1], ids[1:]):
            counts.setdefault(A, {})
            counts[A][B] = counts[A].get(B, 0) + 1
            row[A] = row.get(A, 0) + 1
    return stoi, counts, row

# --- Eval: perplexity (Laplace α) + top-1 next-token accuracy ---
def evaluate(texts, stoi, counts, row, alpha=0.5, topk=5):
    """
    Evaluate a bigram language model on a list of texts.

    Returns a triple (ppl, top1_acc, topk_acc).
    """
    V, a, oov = len(stoi), alpha, stoi[OOV]
    tlp = 0.0  # total log probability
    T = 0       # number of bigram predictions
    correct1 = 0
    correctk = 0
    for t in texts:
        w = tok(t)
        if not w:
            continue
        ids = [stoi.get(x, oov) for x in [BOS, *w, EOS]]
        for A, B in zip(ids[:-1], ids[1:]):
            c = counts.get(A, {}).get(B, 0)
            r = row.get(A, 0)
            tlp += math.log(c + a) - math.log(r + a * V)
            T += 1

            rowA = counts.get(A, {})
            if rowA:
                # top‑1 prediction
                pred1 = max(rowA, key=rowA.get)
                # top‑k candidates
                top_candidates = sorted(rowA.items(), key=lambda x: x[1], reverse=True)[:topk]
                top_preds = {tok_id for tok_id, _ in top_candidates}
            else:
                pred1 = oov
                top_preds = {oov}

            if pred1 == B:
                correct1 += 1
            if B in top_preds:
                correctk += 1

    ppl = math.exp(-tlp / T) if T else float("inf")
    acc1 = (correct1 / T) if T else 0.0
    acck = (correctk / T) if T else 0.0
    return ppl, acc1, acck

# --- Choose a text column by preference, else fallback to first object column ---
def choose_col(df, preferred):
    for c in preferred:
        if c in df.columns: return c
    for c in df.columns:
        if c.lower() != "lang" and df[c].dtype == object: return c
    return df.columns[0]

def main():
    # ---- Load the dataset (your exact style) ----
    splits = {'train': 'train.parquet', 'validation': 'validation.parquet'}
    df_train = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["train"], engine='fastparquet')
    df_val   = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["validation"], engine='fastparquet')

    # Train sets on languages
    df_train_ar = df_train[df_train['lang'] == 'ar']
    df_train_ko = df_train[df_train['lang'] == 'ko']
    df_train_te = df_train[df_train['lang'] == 'te']

    # Validation sets on languages
    df_val_ar = df_val[df_val['lang'] == 'ar']
    df_val_ko = df_val[df_val['lang'] == 'ko']
    df_val_te = df_val[df_val['lang'] == 'te']

    #english contexts
    df_train_enctx = df_train
    df_val_enctx   = df_val

    # Tasks: AR/KO/TE use questions; English contexts use the context column over ALL rows
    tasks = [
        ("Arabic questions",  df_train_ar, df_val_ar, ["question"]),
        ("Korean questions",  df_train_ko, df_val_ko, ["question"]),
        ("Telugu questions",  df_train_te, df_val_te, ["question"]),
        ("English contexts",  df_train_enctx, df_val_enctx, ["context"]),
    ]

    rows = []
    for name, dtr, dva, pref in tasks:
        if dtr.empty: continue
        col_tr = choose_col(dtr, pref)
        col_va = choose_col(dva, pref) if not dva.empty else col_tr
        train = [s for s in dtr[col_tr].astype(str).tolist() if s.strip()]
        val   = [s for s in dva[col_va].astype(str).tolist() if s.strip()] if not dva.empty else []
        stoi, counts, row = fit(train)
        ppl, acc1, acc5 = evaluate(val, stoi, counts, row, alpha=0.5, topk=5)
        rows.append((name, len(stoi), len(train), len(val), ppl, acc1, acc5))

    #Summary only
    print("Results of Bi-gram LM:")
    print(f"{'Model':20s} {'Vocab':>6s} {'TrainN':>7s} {'ValN':>7s} "
          f"{'ValPPL':>10s} {'Top1Acc':>8s} {'Top5Acc':>8s}")
    for name, V, ntr, nva, ppl, acc1, acc5 in rows:
        print(f"{name:20s} {V:6d} {ntr:7d} {nva:7d} {ppl:10.3f} "
              f"{acc1:8.3f} {acc5:8.3f}")

if __name__ == "__main__":
    main()


Results of Bi-gram LM:
Model                 Vocab  TrainN    ValN     ValPPL  Top1Acc  Top5Acc
Arabic questions       5404    2558     415   1100.401    0.244    0.413
Korean questions       4396    2422     356    963.478    0.387    0.474
Telugu questions       2414    1355     384    646.866    0.356    0.484
English contexts      83615   15343    3011   6847.933    0.178    0.316
